# Prediction Decomposition Analysis

Break down the model's final predictions by component: per-layer
logit contributions, direct logit attribution, entropy decomposition,
and component agreement.

## Why This Matters

Composition prediction decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.prediction_decomposition_analysis import (
    logit_contribution_by_layer, direct_logit_attribution,
    prediction_entropy_decomposition, component_prediction_agreement,
    prediction_decomposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Logit Contribution by Layer

How much does each layer's attention and MLP contribute to final logits?

In [ ]:
result = logit_contribution_by_layer(model, tokens, position=-1)
print(f"Top predicted tokens: {result['top_tokens']}")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_contribution'] * 2)
    mlp_bar = '█' * int(p['mlp_contribution'] * 2)
    print(f"  Layer {p['layer']}: attn={p['attn_contribution']:.3f} {attn_bar}")
    print(f"           mlp={p['mlp_contribution']:.3f} {mlp_bar}")

## Direct Logit Attribution

Which specific components drive the predicted token's logit?

In [ ]:
result = direct_logit_attribution(model, tokens, position=-1)
print(f"Target token: {result['target_token']}")
print(f"Total logit: {result['total_logit']:.4f}")
for c in result['per_component'][:8]:
    bar = '█' * int(abs(c['contribution']) * 10)
    sign = '+' if c['contribution'] > 0 else '-'
    print(f"  {c['component']:12s}: {sign}{abs(c['contribution']):.4f} {bar}")

## Entropy Decomposition

How does each layer affect prediction uncertainty?

In [ ]:
result = prediction_entropy_decomposition(model, tokens, position=-1)
print(f"Total entropy reduction: {result['total_entropy_reduction']:.4f}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f} "
          f"change={p['entropy_change']:+.4f}")

## Component Prediction Agreement

Do attention and MLP within each layer agree on predictions?

In [ ]:
result = component_prediction_agreement(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: top-{result['top_k']} overlap={p['top_k_overlap']:.0%} "
          f"cos={p['logit_cosine']:.4f}")

## Decomposition Summary

In [ ]:
result = prediction_decomposition_summary(model, tokens, position=-1)
print(f"Top tokens: {result['top_tokens']}")
print(f"Total entropy reduction: {result['total_entropy_reduction']:.4f}")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: H={p['entropy']:.4f} agree={p['agreement']:.0%}")